# 04 - Model Training (Baseline) dan Validasi Fitur
**Proyek:** Hotel Booking Cancellation Prediction | **Tahap:** M5 | **PRD:** v1.1

Isi: (1) baseline 3 algoritma dengan 5-fold stratified cross-validation **pada data train saja**, (2) validasi ulang pemilihan 5 input, (3) menyimpan model baseline.
**Data test tidak disentuh** di notebook ini; dipakai sekali di M6 untuk evaluasi akhir.

Library: `pip install scikit-learn xgboost joblib pandas numpy matplotlib seaborn`

In [ ]:
import os, glob, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
os.makedirs('models', exist_ok=True); os.makedirs('reports/figures', exist_ok=True)

# --- Muat data mentah, hapus duplikat, buat fitur turunan (sama dengan M4) ---
cands = (glob.glob('/kaggle/input/**/hotel_bookings.csv', recursive=True) + glob.glob('./hotel_bookings.csv')
         + glob.glob('./data/raw/hotel_bookings.csv') + glob.glob('/mnt/user-data/uploads/hotel_bookings.csv'))
assert cands, 'hotel_bookings.csv tidak ditemukan'
raw = pd.read_csv(cands[0])
df = raw.drop_duplicates(subset=[c for c in raw.columns if c != 'reservation_status_date']).reset_index(drop=True)
df['nights'] = df.stays_in_weekend_nights + df.stays_in_week_nights
df['guests'] = df.adults + df.children.fillna(0) + df.babies
TARGET = 'is_canceled'

# --- Split identik dengan M4 (80:20, stratified, random_state=42). Test TIDAK dipakai di notebook ini. ---
train_idx, test_idx = train_test_split(df.index, test_size=0.2, stratify=df[TARGET], random_state=RANDOM_STATE)
train = df.loc[train_idx]
y_train = train[TARGET]
print('Train:', train.shape, f'| persen batal: {y_train.mean()*100:.2f}')

# Konsistensi dengan M4: indeks train harus sama dengan data/processed/train.csv (jika file ada)
if os.path.exists('data/processed/train.csv'):
    saved = pd.read_csv('data/processed/train.csv', index_col=0).index
    print('Split sama dengan M4:', set(saved) == set(train_idx))

## 1. Preprocessing builder dan daftar model

In [ ]:
NUM_ALL = ['lead_time','total_of_special_requests','nights','guests','adr']
log_tf = FunctionTransformer(np.log1p, feature_names_out='one-to-one')

def build_prep(feats, linear=False):
    """ColumnTransformer untuk set fitur tertentu. linear=True -> log1p(lead_time) + scaling."""
    nums = [f for f in feats if f in NUM_ALL]
    cats = [f for f in feats if f not in NUM_ALL]
    tfs = []
    if 'country' in cats:
        tfs.append(('country', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='Unknown')),
                    ('oh', OneHotEncoder(handle_unknown='infrequent_if_exist', max_categories=11, sparse_output=False))]), ['country']))
        cats = [c for c in cats if c != 'country']
    if cats:
        tfs.append(('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cats))
    if linear:
        for n in nums:
            steps = ([('log', log_tf)] if n == 'lead_time' else []) + [('sc', StandardScaler())]
            tfs.append((n, Pipeline(steps), [n]))
    elif nums:
        tfs.append(('num', 'passthrough', nums))
    return ColumnTransformer(tfs)

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
def get_models():
    return {
     'Dummy (kelas mayoritas)':        (DummyClassifier(strategy='most_frequent'), False),
     'LogReg':                         (LogisticRegression(max_iter=1000), True),
     'LogReg (balanced)':              (LogisticRegression(max_iter=1000, class_weight='balanced'), True),
     'RandomForest':                   (RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE), False),
     'RandomForest (balanced)':        (RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE), False),
     'XGBoost':                        (XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                                                      eval_metric='logloss', n_jobs=-1, random_state=RANDOM_STATE), False),
     'XGBoost (scale_pos_weight)':     (XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                                                      scale_pos_weight=pos_weight, eval_metric='logloss', n_jobs=-1, random_state=RANDOM_STATE), False),
    }

def make_pipe(feats, name):
    model, linear = get_models()[name]
    return Pipeline([('prep', build_prep(feats, linear)), ('model', model)])

FEATS_5 = ['lead_time','country','market_segment','total_of_special_requests','customer_type']
print('Fitur 5 input:', FEATS_5, '| rasio kelas (neg/pos) =', round(pos_weight, 2))

**Alasan pemilihan model (siapkan untuk tanya jawab):**
- **Dummy (kelas mayoritas):** pembanding paling dasar. Model apa pun harus mengalahkannya agar dianggap belajar sesuatu.
- **Logistic Regression:** baseline linear yang sederhana dan mudah ditafsirkan (koefisien). Butuh scaling.
- **Random Forest:** ensemble *bagging* yang menangkap hubungan non-linear dan interaksi antarfitur (misalnya `country` × `market_segment`) tanpa scaling.
- **XGBoost:** ensemble *boosting* yang memperbaiki kesalahan secara bertahap; biasanya kuat pada data tabular.
- **Varian *balanced* / `scale_pos_weight`:** karena kelas batal hanya ≈27%, setiap model diuji dengan dan tanpa pembobotan kelas untuk melihat dampaknya pada Recall.

## 2. Baseline: 5-fold stratified CV pada data train

In [ ]:
SCORING = ['accuracy','precision','recall','f1','roc_auc','average_precision']
skf5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name in get_models():
    t0 = time.time()
    cv = cross_validate(make_pipe(FEATS_5, name), train[FEATS_5], y_train, cv=skf5, scoring=SCORING,
                        return_train_score=True, n_jobs=1)
    row = {'model': name}
    for m in SCORING:
        row[m] = cv[f'test_{m}'].mean()
    row['f1_train'] = cv['train_f1'].mean(); row['auc_train'] = cv['train_roc_auc'].mean()
    row['gap_auc'] = row['auc_train'] - row['roc_auc']
    row['detik'] = time.time() - t0
    rows.append(row)
    print(f"{name:30s} selesai ({row['detik']:.0f} dtk)")

res = pd.DataFrame(rows).set_index('model').round(3)
res.to_csv('reports/baseline_cv_results.csv')
res

**Interpretasi hasil baseline (5-fold CV, data train, threshold 0,5):**
- **Accuracy menyesatkan:** model Dummy yang selalu menebak "tidak batal" sudah mendapat accuracy 0,727 tetapi Recall 0. Karena itu model dibandingkan dengan Recall, F1, ROC-AUC, dan PR-AUC.
- **Model terbaik (kemampuan memisahkan kelas):** XGBoost, dengan ROC-AUC 0,833 dan PR-AUC 0,646. Random Forest di bawahnya (0,825 dan 0,628). **Terendah:** Logistic Regression (ROC-AUC 0,801, PR-AUC ≈0,59), tetapi masih jauh di atas Dummy (0,500 dan 0,273). Pola ini menunjukkan ada hubungan non-linear/interaksi yang tidak tertangkap model linear.
- **Efek pembobotan kelas:** tanpa bobot, XGBoost hanya menemukan 51,7% pembatalan (Recall) dengan Precision 0,652. Dengan `scale_pos_weight`, Recall naik ke 0,760 tetapi Precision turun ke 0,524 dan accuracy turun dari 0,793 ke 0,746. Jadi ada *trade-off*: lebih banyak pembatalan terdeteksi, tetapi lebih banyak peringatan yang keliru. ROC-AUC/PR-AUC hampir tidak berubah karena pembobotan hanya menggeser titik keputusan, bukan kemampuan memisahkan kelas.
- **Overfitting:** selisih AUC train vs validasi: Logistic Regression 0,000 (tidak overfit, tetapi kapasitasnya terbatas), XGBoost ≈0,025-0,027, Random Forest ≈0,058-0,060 (overfitting ringan). Perlu dikendalikan dengan tuning (`max_depth`, `min_samples_leaf`) di M6.
- **Waktu:** Random Forest paling lambat (≈40 dtk untuk 5-fold), XGBoost ≈8 dtk.

In [ ]:
# Visualisasi perbandingan model (metrik validasi)
plot_df = res.drop(index='Dummy (kelas mayoritas)')[['precision','recall','f1','roc_auc','average_precision']]
ax = plot_df.plot(kind='bar', figsize=(12,4.5)); ax.set_title('Perbandingan baseline (5-fold CV, data train, threshold 0,5)')
ax.set_ylabel('skor'); plt.xticks(rotation=25, ha='right'); plt.tight_layout()
plt.savefig('reports/figures/baseline_comparison.png', dpi=120); plt.show()

**Interpretasi grafik:** batang tertinggi pada ROC-AUC dan PR-AUC dimiliki XGBoost. Varian berbobot (*balanced*) memiliki Recall tinggi tetapi Precision lebih rendah, dan sebaliknya untuk varian tanpa bobot. Pemilihan akhir bergantung pada tujuan: jika ingin menemukan sebanyak mungkin pembatalan, Recall diutamakan; jika ingin menghindari peringatan keliru, Precision diutamakan. Keputusan ini diambil di M6.

## 3. Validasi ulang pemilihan 5 input
Membandingkan beberapa set fitur dengan **3 model** memakai 3-fold CV pada data train. Metrik utama: **ROC-AUC** dan **PR-AUC (average precision)**, yang tidak bergantung pada threshold. Fitur kandidat tambahan: `hotel`, `deposit_type`, `arrival_date_month`, `nights`, `guests`, `adr`.

In [ ]:
SETS = {
 'A. 6 input PRD v1.0':      ['hotel','lead_time','arrival_date_month','nights','guests','deposit_type'],
 'B. 5 input (usulan)':      FEATS_5,
 'C. 5 input, adr ganti customer_type': ['lead_time','country','market_segment','total_of_special_requests','adr'],
 'D. 7 input (B + adr + hotel)':        FEATS_5 + ['adr','hotel'],
}
MODELS3 = ['LogReg (balanced)', 'RandomForest', 'XGBoost']
skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
ablation = []
for sname, feats in SETS.items():
    for mname in MODELS3:
        cv = cross_validate(make_pipe(feats, mname), train[feats], y_train, cv=skf3,
                            scoring=['roc_auc','average_precision'], n_jobs=1)
        ablation.append({'set fitur': sname, 'n_input': len(feats), 'model': mname,
                         'ROC-AUC': cv['test_roc_auc'].mean(), 'PR-AUC': cv['test_average_precision'].mean()})
    print(sname, 'selesai')
abl = pd.DataFrame(ablation).round(3)
display(abl.pivot(index='set fitur', columns='model', values='ROC-AUC').assign(n_input=[len(SETS[k]) for k in sorted(SETS)]))
display(abl.pivot(index='set fitur', columns='model', values='PR-AUC'))
abl.to_csv('reports/feature_set_validation.csv', index=False)

**Interpretasi validasi fitur:**
- **Set A (6 input PRD awal) paling lemah** pada ketiga model (ROC-AUC 0,67-0,70; PR-AUC 0,44-0,48). Ini mengonfirmasi bahwa keputusan mengganti 6 input awal dengan 5 input baru tepat.
- **Set B (5 input usulan):** ROC-AUC 0,80-0,83 dan PR-AUC 0,59-0,64, naik tajam dibanding A pada semua model.
- **Set C (adr menggantikan `customer_type`):** hampir identik dengan B (XGBoost 0,835 vs 0,832), sehingga `customer_type` tetap dipilih karena lebih mudah diisi pengguna (dropdown 4 pilihan).
- **Set D (7 input):** naik lagi tetapi modest: XGBoost ROC-AUC 0,832 → 0,849 dan PR-AUC 0,644 → 0,676; Random Forest 0,823 → 0,845. Gain sekitar 0,02 ROC-AUC dengan menambah 2 isian formulir.
- Urutan model konsisten di semua set fitur (XGBoost ≥ Random Forest > Logistic Regression), sehingga kesimpulan perbandingan model tidak bergantung pada pilihan fitur.
- **Keputusan:** **tetap 5 input** (kesederhanaan form vs gain 0,02). Jika ingin akurasi sedikit lebih tinggi, tambahkan `adr` dan `hotel` sebagai opsi. Keputusan akhir dicatat di PRD.

## 4. Melatih model baseline pada seluruh data train dan menyimpannya

In [ ]:
saved = {}
for name in ['LogReg (balanced)', 'RandomForest', 'XGBoost']:
    pipe = make_pipe(FEATS_5, name).fit(train[FEATS_5], y_train)
    fname = 'models/baseline_' + name.split()[0].lower() + '.joblib'
    joblib.dump(pipe, fname); saved[name] = fname
print(saved)

# Cek: model tersimpan bisa dimuat dan memprediksi satu reservasi contoh
sample = pd.DataFrame([{'lead_time': 120, 'country': 'PRT', 'market_segment': 'Online TA',
                        'total_of_special_requests': 0, 'customer_type': 'Transient'}])
for name, f in saved.items():
    print(f'{name:20s} P(batal) = {joblib.load(f).predict_proba(sample)[0,1]:.3f}')

**Interpretasi:** untuk satu reservasi contoh yang sama, probabilitas batal berbeda antar model (Logistic Regression berbobot ≈0,94, Random Forest ≈0,67, XGBoost ≈0,78). Ini wajar: pembobotan kelas menaikkan probabilitas secara sistematis, sehingga angkanya **belum terkalibrasi** dan tidak boleh ditampilkan di aplikasi sebagai "risiko pasti". Kalibrasi probabilitas dan pemilihan threshold dievaluasi di M6.

## Kesimpulan M5
1. Ketiga model mengalahkan Dummy dengan jelas; **XGBoost terbaik** (ROC-AUC CV 0,833, PR-AUC 0,646), Logistic Regression terendah (0,801 / ≈0,59), Random Forest di tengah dengan overfitting ringan.
2. Pembobotan kelas menaikkan Recall (≈0,72-0,76) dengan mengorbankan Precision dan accuracy; ranking model tidak berubah.
3. Validasi ulang: **5 input** jauh lebih baik dari 6 input awal; menambah `adr` dan `hotel` hanya memberi gain ≈0,02 ROC-AUC.
4. Tiga model baseline tersimpan di `models/`; hasil CV di `reports/baseline_cv_results.csv` dan `reports/feature_set_validation.csv`.
5. Data test belum dipakai.

**Langkah berikutnya: M6 Tuning dan Evaluasi** (`05_model_evaluation.ipynb`):
- Tuning terkontrol (RandomizedSearchCV pada data train) untuk ketiga model.
- Pemilihan threshold lewat cross-validation di data train (bukan test).
- **Evaluasi akhir sekali pada data test:** Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC, confusion matrix, kurva ROC/PR.
- Analisis overfitting, kesalahan FP/FN, feature importance, dan alasan pemilihan model final.